In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from scipy import stats

import warnings
warnings.filterwarnings("ignore")

# Principal Component Analysis (PCA) & Chi-Square Test

## Objective

Reduce the five correlated operational variables:

- actual_power_kW
- wind_speed_m/s
- theoretical_power_kW
- power_delta
- eff

to a smaller number of principal components while retaining at least 90% of the total variance.

Temporal variables (month, year, hour) and wind direction are excluded because they do not represent continuous operational measurements of the wind turbine.

## Research Questions

1. Can the operational variables be reduced while preserving most of the information?

2. Is turbine performance independent of wind direction?

In [2]:
df = pd.read_csv("../data/processed/wind_turbine.csv")

print("Shape :", df.shape)

df.head()

Shape : (42781, 13)


,date_time,actual_power_kW,wind_speed_m/s,theoretical_power_kW,wind_direction_deg,month,year,hour,power_delta,eff,season,wind_dir,performance_flag
0,2018-01-01 00:00:00,380.047791,5.311336,416.328908,259.994904,1,2018,0,36.281117,0.912855,Winter,W,Normal
1,2018-01-01 00:10:00,453.769196,5.672167,519.917511,268.641113,1,2018,0,66.148316,0.872772,Winter,W,Normal
2,2018-01-01 00:20:00,306.376587,5.216037,390.900016,272.564789,1,2018,0,84.523429,0.783772,Winter,W,Underperforming
3,2018-01-01 00:30:00,419.645905,5.659674,516.127569,271.258087,1,2018,0,96.481664,0.813066,Winter,W,Normal
4,2018-01-01 00:40:00,380.650696,5.577941,491.702972,265.674286,1,2018,0,111.052276,0.774148,Winter,W,Underperforming


In [3]:
pca_features = [
    "actual_power_kW",
    "wind_speed_m/s",
    "theoretical_power_kW",
    "power_delta",
    "eff"
]

x = df[pca_features]

x.head()

,actual_power_kW,wind_speed_m/s,theoretical_power_kW,power_delta,eff
0,380.047791,5.311336,416.328908,36.281117,0.912855
1,453.769196,5.672167,519.917511,66.148316,0.872772
2,306.376587,5.216037,390.900016,84.523429,0.783772
3,419.645905,5.659674,516.127569,96.481664,0.813066
4,380.650696,5.577941,491.702972,111.052276,0.774148


## Principal Component Analysis (PCA)

In [4]:
# Standardize the selected variables
x_std = StandardScaler().fit_transform(x)

# Compute covariance matrix
cov_mat = np.cov(x_std, rowvar=False)

# Compute eigenvalues and eigenvectors
eigen_values, eigen_vectors = np.linalg.eig(cov_mat)

# Sort components by explained variance
sort_index = np.argsort(eigen_values)[::-1]

eigen_values = eigen_values[sort_index]
eigen_vectors = eigen_vectors[:, sort_index]

# Calculate explained and cumulative variance
explained_variance = eigen_values / np.sum(eigen_values)
cumulative_variance = np.cumsum(explained_variance)

# Retain components explaining at least 90% variance
variance_threshold = 0.90

for i in range(len(eigen_values)):
    print(
        f"PC{i+1} : Explained = {explained_variance[i]:.4f} | "
        f"Cumulative = {cumulative_variance[i]:.4f}"
    )

for i in range(len(eigen_values)):
    if cumulative_variance[i] >= variance_threshold:
        k = i + 1
        break

# Compute principal components
principal_components = eigen_vectors[:, :k]

# Transform original data
x_pca = x_std @ principal_components

print("Original Shape :", x.shape)
print("Reduced Shape :", x_pca.shape)
print("Number of Components :", k)

PC1 : Explained = 0.6065 | Cumulative = 0.6065
PC2 : Explained = 0.2974 | Cumulative = 0.9039
PC3 : Explained = 0.0799 | Cumulative = 0.9838
PC4 : Explained = 0.0162 | Cumulative = 1.0000
PC5 : Explained = 0.0000 | Cumulative = 1.0000
Original Shape : (42781, 5)
Reduced Shape : (42781, 2)
Number of Components : 2


In [6]:
# Principal component loadings
loadings = pd.DataFrame(
    principal_components,
    index=pca_features,
    columns=[f"PC{i+1}" for i in range(k)]
)

print("Principal Component Loadings")
print(loadings.round(4))

Principal Component Loadings
                         PC1     PC2
actual_power_kW      -0.5622 -0.0521
wind_speed_m/s       -0.5465  0.1564
theoretical_power_kW -0.5505  0.2119
power_delta           0.0031  0.7582
eff                  -0.2869 -0.5942


In [ ]:
df["PC1"] = x_pca[:, 0]
df["PC2"] = x_pca[:, 1]

In [9]:
# Columns to export for Tableau
tableau_cols = [
    "date_time",
    "year",
    "month",
    "hour",
    "season",
    "wind_speed_m/s",
    "wind_direction_deg",
    "wind_dir",
    "actual_power_kW",
    "theoretical_power_kW",
    "power_delta",
    "eff",
    "performance_flag",
    "PC1",
    "PC2"
]

In [10]:
# Export dataset for Tableau
df[tableau_cols].to_csv(
    "../data/processed/turbine_with_pca.csv",
    index=False
)

## Chi-Square Test of Independence

### Objective

Determine whether turbine performance is associated with wind direction.

### Hypotheses

H0 : Turbine performance is independent of wind direction.

H1 : Turbine performance depends on wind direction.

alpha = 0.05

In [11]:
# Create contingency table
ct = pd.crosstab(
    df["wind_dir"],
    df["performance_flag"]
)

print("Contingency Table")
print(ct)

Contingency Table
performance_flag  Normal  Underperforming
wind_dir                                 
E                   4187             2103
N                   1821             1074
NE                 13839             3912
NW                   422              444
S                   5503              904
SE                   545              384
SW                  4663             1004
W                   1106              870


In [12]:
# Perform Chi-Square Test
ts, p, dof, exp = stats.chi2_contingency(ct)

exp_df = pd.DataFrame(
    exp,
    index=ct.index,
    columns=ct.columns
)

print("Chi-Square Statistic :", round(ts, 4))
print("P-value :", round(p, 4))
print("Degrees of Freedom :", dof)

print("\nExpected Frequencies")
print(exp_df.round(2))

Chi-Square Statistic : 1945.8355
P-value : 0.0
Degrees of Freedom : 7

Expected Frequencies
performance_flag    Normal  Underperforming
wind_dir                                   
E                  4717.54          1572.46
N                  2171.27           723.73
NE                13313.35          4437.65
NW                  649.51           216.49
S                  4805.29          1601.71
SE                  696.76           232.24
SW                 4250.28          1416.72
W                  1482.01           493.99


In [23]:
# % underperforming by direction 
ct_pct = pd.crosstab( df['wind_dir'], df['performance_flag'],  normalize='index') * 100
ct_pct.sort_values('Underperforming')
print(ct_pct.round(2).sort_values('Underperforming',ascending=False))

performance_flag  Normal  Underperforming
wind_dir                                 
NW                 48.73            51.27
W                  55.97            44.03
SE                 58.67            41.33
N                  62.90            37.10
E                  66.57            33.43
NE                 77.96            22.04
SW                 82.28            17.72
S                  85.89            14.11


### Decision

If p-value < 0.05, reject H0.

Otherwise, fail to reject H0.

In [13]:
if p < 0.05:
    print("Conclusion : Reject H0")
    print("Wind direction and turbine performance are associated.")
else:
    print("Conclusion : Fail to Reject H0")
    print("Wind direction and turbine performance are independent.")

Conclusion : Reject H0
Wind direction and turbine performance are associated.


### Underperforming Percentage by Wind Direction

In [14]:
# Percentage of underperforming records by wind direction
ct_pct = pd.crosstab(
    df["wind_dir"],
    df["performance_flag"],
    normalize="index"
) * 100

ct_pct = ct_pct.round(2).sort_values(
    "Underperforming",
    ascending=False
)

print(ct_pct)

performance_flag  Normal  Underperforming
wind_dir                                 
NW                 48.73            51.27
W                  55.97            44.03
SE                 58.67            41.33
N                  62.90            37.10
E                  66.57            33.43
NE                 77.96            22.04
SW                 82.28            17.72
S                  85.89            14.11


In [15]:
wind_distribution = (
    df["wind_dir"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Wind Direction Distribution (%)")
print(wind_distribution)

Wind Direction Distribution (%)
wind_dir
NE    41.49
S     14.98
E     14.70
SW    13.25
N      6.77
W      4.62
SE     2.17
NW     2.02
Name: proportion, dtype: float64


### Interpretation

The Chi-Square Test determines whether turbine performance is associated with wind direction. The percentage table highlights which wind directions have higher or lower underperforming rates, while the wind direction distribution indicates the most frequently occurring wind directions in the dataset.